# Scenario One: Salinity and Freshwater Vegetation

## Description
This jupyter notebook explores the relationship between two week salinity levels and the presence of freshwater marsh vegetation in Louisiana coastal areas. The notebook aims to demonstrate how to pull data from the Master Plan API to visualize and analyze the outputs of the ICM, CLARA, and PT models. The data pulled is as follows:
1. Two week salinity levels from the ___ model 
2. Freshwater Marsh Vegetation data from the ___ model 
3. Vector Boundaries of Grid Cells 
4. Raster Crosswalk Data 

*Resources:* 

[Master Plan API Documentation](https://github.com/pscedu/cpra.mp.data/tree/main)

In [ ]:
import polars as pl
from polars import col
import geopandas as gpd
import rasterio as rio
import numpy as np
import time
from cpra.mp.data import read_data
from matplotlib import pyplot as plt
from rasterio.plot import show
from rasterio.features import shapes
import altair as alt
import matplotlib.colors
import json

### Establish Variable Inputs, Outputs, and Switches

In [ ]:
VERSION = "v4"  # NOTE - Incorporating range of years
SAVE_FILES = True
RUN_TEST = False
single_year = 2019
scenario = 7
model_group_id = 500
salinity_variable = "sal"
freshwater_marsh_variable = "pl_fm" # percent land freshwater marsh 
hydrocompartment_id_column = "hydrocompartment_id"
freshwater_marsh_value_column = "freshwater_marsh_value"
calendar_year = 'calendar_year'
SALINITY_THRESHOLD = 5.5
FRESHWATER_MARSH_THRESHOLD = 0.5

In [ ]:
# Input and Output Files
crosswalk_veg_cell_hydro_compartment = "/ocean/projects/bcs200002p/shared/grids/crosswalks/veg_grid_cell_v001__hydro_compartment_v001.tif"
vector_boudndaries_shp = '/ocean/projects/bcs200002p/mbrennan/MP2023_S00_G000_C000_U00_V00_SLA_I_00_00_H_comps.shp/MP2023_S00_G000_C000_U00_V00_SLA_I_00_00_H_comps.shp'
# ^ to be changed to new location 

OUTPUT_SALINITY_CSV = "{}_{}_{}_{}_long_{}.csv".format(
    salinity_variable, single_year, scenario, model_group_id, VERSION)

OUTPUT_TIFF = "threshold_analysis_numpy_{}.tif".format(VERSION)
OUTPUT_GEOJSON = "sal_map_{}_{}.geojson".format(single_year, VERSION)

#### Step One: Load Two Week Salinity Data 

Using the Master Plan API, pull two week salinity data for the selected scenario and time period. This data is returned in a wide format, where the hydrocompartment ids are all column headers. In order to use this data for analysis, it must be transformed into a long format and desired columns selected. 

In [ ]:
salinity_lf = read_data(
    variable=salinity_variable,
    grid="hydro_compartment_v001",
    time_unit="annual",
    model_group_id=model_group_id,
    scenario_id=scenario,
    aggregate_type="2_wk_max",
)

# Identify id columns and value columns (numeric fids are the rest)
id_cols = [
    "variable",
    "grid",
    "time_unit",
    "model_group_id",
    "scenario_id",
    "aggregate_type",
    "calendar_year",
    "model"
]

value_cols = [c for c in salinity_lf.collect_schema() if c not in id_cols]

# Make Salinity Data Long
sal_long = (
    (
        salinity_lf.unpivot(
            on=value_cols,
            index=id_cols,
            variable_name=hydrocompartment_id_column,
            value_name=salinity_variable,
        ).with_columns([
            col(hydrocompartment_id_column).cast(pl.Int64),
            col(salinity_variable).cast(pl.Float64),
            col(calendar_year).cast(pl.Int64),
        ])
    ).collect()
)

if SAVE_FILES:
    sal_long.write_csv(OUTPUT_SALINITY_CSV)

sal_selected = sal_long.select(
    [hydrocompartment_id_column, "calendar_year", "sal", "aggregate_type"]
)

sal_selected

#### Step Two: Load Freshwater Marsh Vegetation Data

Freshwater marsh vegetation is also loaded using the Master Plan API with the stored scenario, model, and time period variables. The paths to the raster data is stored in a column named 'path,' a dictionary is created to store the raster data for each year to loop through later in the notebook.  

In [ ]:
freshwater_marsh_tif_df = (
    read_data(
    variable="ffibs_coverage_ratio",
    grid="veg_grid_cell_v001",
    time_unit="annual",
    model_group_id=model_group_id,
    scenario_id=scenario,
    ffibs_type=freshwater_marsh_variable,
    ))

freshwater_marsh_tif_dict = dict(freshwater_marsh_tif_df.collect().select("calendar_year", "path").iter_rows())

### Step Three: Data Analysis

The following section joins the salinity and vegetation data together using rasterio and numpy to create windowed lookup arrays. This allows for efficient extraction of salinity values for each veg grid cell (and corresponding hydrocompartment id) that contains freshwater marsh vegetation. 

#### References: 

[Rasterio Window Documentation](https://rasterio.readthedocs.io/en/stable/topics/windowed-rw.html)

In [ ]:
max_id = sal_selected[hydrocompartment_id_column].max() # Get maximum hydrocompartment id for lookup array size

salinity_lookup = np.full(max_id + 1, np.nan) # Initialize lookup array with NaNs

with rio.open(crosswalk_veg_cell_hydro_compartment) as CROSSWALK:
    
    meta = CROSSWALK.meta.copy()

    print(f"\nVEG HYDRO CROSSWALK METADATA: {meta}")
    
    meta.update(dtype="float32", count=1, compress="zstd", nodata=-9999) 
    
    # [x] Updated nodata to -9999 to match standards, updated compression to zstd
    # TODO Triple check against harrison's and matt's standards
    
    with rio.open(OUTPUT_TIFF, "w", **meta) as salinity_raster: # Create output raster with updated metadata
        for block_index, window in CROSSWALK.block_windows(1):
            windowed_year_dictionary = {}
            window_year_list = []
            
            print(f"On block_index {block_index} for Crosswalk Window...")
        
            for year, tif in freshwater_marsh_tif_dict.items(): # YEAR BEFORE VEG DATA, YEAR AFTER SALINITY 
                # print(f"Processing year {year} with TIF file: {tif}")
                with rio.open(tif) as FRESHWATER_MARSH_VALUES:
                    hydrocompartment_ids = CROSSWALK.read(1, window=window).flatten()
                    freshwater_marsh_values_arr = FRESHWATER_MARSH_VALUES.read(1, window=window).flatten()

                    next_year = min(year + 1, 2070)
                    
                    salinity_data = sal_selected.filter(
                        (col("calendar_year") == next_year)
                        & (col(hydrocompartment_id_column).is_in(hydrocompartment_ids)))
            
                    ids = salinity_data[hydrocompartment_id_column].to_numpy()
                    values = salinity_data[salinity_variable].to_numpy()
            
                    salinity_lookup[ids] = values
            
                    mapped_salinity_arr = np.where(
                        hydrocompartment_ids == -9999,
                        np.nan,  # Assign NaN where ID is -9999
                        salinity_lookup[
                            np.clip(hydrocompartment_ids, 0, max_id).astype(int) 
                        ],  # Lookup for all else
                    )
            
                    if RUN_TEST:
                        test_row = salinity_data.sample(1)
                        test_id = test_row[hydrocompartment_id_column].item()
                        expected_salinity = test_row[salinity_variable].item()
            
                        coords = np.where(hydrocompartment_ids == test_id)
            
                        actual_values = mapped_salinity_arr[coords]
            
                        if np.allclose(actual_values, expected_salinity, equal_nan=True):
                            print(f"Success! ID {test_id} correctly mapped to {expected_salinity}.")
                        else:
                            print("Error: Mismatched values found.")
            
                    threshold_condition = (mapped_salinity_arr > SALINITY_THRESHOLD) & ((freshwater_marsh_values_arr > FRESHWATER_MARSH_THRESHOLD) & (freshwater_marsh_values_arr > 0))
            
                    threshold_raster = np.where(
                        np.isnan(mapped_salinity_arr),
                        np.nan, 
                        np.where(threshold_condition, year, 0)
                        )
                    
                    # NOTE In polars, column names must be strings 
                    window_year_list.append(threshold_raster)
                    salinity_lookup[ids] = np.nan
            
            stacked_arrays = np.stack(window_year_list)
            
            stacked_arrays[stacked_arrays == 0] = np.nan
            
            result_array = np.nanmin(stacked_arrays, axis=0).reshape(window.height, window.width)
            
            print(f"Lowest Years at Each Index: {np.unique(result_array)}")
            
            if SAVE_FILES:
                salinity_raster.write(result_array, indexes=1, window=window)

### Visualize Results

In [ ]:
# MATPLOTLIB

src = rio.open(OUTPUT_TIFF)

src_arr = src.read(1)

gdf = gpd.read_file(vector_boudndaries_shp)

fig, ax = plt.subplots(figsize=(10, 10))

gdf.plot(ax=ax, color='none', edgecolor='lightgray', linewidth=0.25)

years = np.unique(src_arr[src_arr > 0]) 

cmap = plt.get_cmap('viridis', len(years))
im = show(src, ax=ax, transform=src.transform, cmap=cmap, title=f'{salinity_variable} x {freshwater_marsh_variable}: Threshold Analysis Results')
# im = ax.get_images()[0]
# cbar = fig.colorbar(im, ax=ax, ticks=years)
# cbar.set_label('Year')

# plt.xlabel("Easting")
# plt.ylabel("Northing")
plt.show()

# Close the raster dataset
src.close()

In [ ]:
# Convert TRESHOLD ANALYSIS TIFF to GEOJSON for visualization in altair
def tiff_to_geojson(input_tiff, output_file, field_name):   
    with rio.open(input_tiff) as src:
        image = src.read(1) 
        transform = src.transform
        crs = src.crs
    
    
    features = []
    for geom, val in shapes(image, transform=transform):
        features.append({
            "type": "Feature",
            "geometry": geom,
            "properties": {field_name: val}
        })
    
    # Create GeoJSON FeatureCollection
    geojson_data = {
        "type": "FeatureCollection",
        "crs": {"type": "name", "properties": {"name": crs.to_string()}},
        "features": features
    }
    
    # Save as GeoJSON 
    with open(output_file, 'w') as f:
        json.dump(geojson_data, f)
        
    return output_file

output_geojson    = tiff_to_geojson(OUTPUT_TIFF, 'threshold_analysis_v4.geojson', calendar_year)
crosswalk_geojson = tiff_to_geojson(crosswalk_veg_cell_hydro_compartment, 'crosswalk_veg_cell_hydro_compartment.geojson', hydrocompartment_id_column)

In [ ]:
threshold_analysis_gdf = gpd.read_file(output_geojson)
threshold_analysis_gdf[calendar_year] = threshold_analysis_gdf[calendar_year].astype('Int64')
threshold_analysis_gdf

In [ ]:
crosswalk_gdf = gpd.read_file(crosswalk_geojson)
crosswalk_gdf[hydrocompartment_id_column] = crosswalk_gdf[hydrocompartment_id_column].astype('Int64')
crosswalk_gdf

In [ ]:
background = alt.Chart(crosswalk_gdf).mark_geoshape(
    stroke='lightgray',    # Set a default stroke color (can be customized),
    fillOpacity=0
).encode(tooltip=[f'{hydrocompartment_id_column}:N']
).project(
    type="identity", reflectY=True
).properties(
    width=700,
    height=400
)
background

In [ ]:
base = alt.Chart(threshold_analysis_gdf).mark_geoshape(fill='transparent', stroke='white').encode(
    tooltip=[f'{calendar_year}:N'])

chart = alt.Chart(threshold_analysis_gdf).mark_geoshape(  # Add thresholds as variable inputs into map titles/legends 
    stroke='transparent',
    strokeWidth=1
).encode(
    color=alt.condition(
        alt.datum.calendar_year == None,
        alt.value('transparent'),
        alt.Color(f'{calendar_year}:N', scale=alt.Scale(scheme='rainbow'))
    ),
    tooltip=[f'{calendar_year}:N'] 
).project(
    type="identity", reflectY=True
).properties(
    title=f'{salinity_variable} and {freshwater_marsh_variable} Threshold Analysis, scenario {scenario}, model group {model_group_id}',
    width=700,
    height=400
).transform_filter(
    f'isValid(datum.{calendar_year})' # Removes nulls from the legend and marks
)

threshold_analysis_display = base + chart
threshold_analysis_display

In [ ]:
overlay = background + threshold_analysis_display

overlay